# 选修E3 · Day 1 上机：Transformer 架构与训练流程

**版本**：v5.0 学习材料包
**配套**：notes.md（讲义）｜ data/README.md（真实库/数据）｜ solution.ipynb（参考答案，做完再看）

## 学习目标
学完你能：
1. 用 **tiktoken** 和 **transformers AutoTokenizer** 对营销文案做真实 tokenization，对比中英文 token 消耗
2. 从 **transformers AutoConfig** 读取 GPT-2 架构参数，推算参数量（~124M）
3. 用 **torch** 手写 Self-Attention 和 Transformer Block，理解架构而非黑箱
4. 在营销文案 token 上运行注意力，可视化注意力矩阵
5. 理解 CLM 预训练任务和训练三阶段（Pre-training/SFT/Alignment）

## 说明
本笔记本有 **6 个 TODO**，你需要自己填写代码。每个 TODO 有提示。
真实库：transformers（config+tokenizer）+ torch（手写注意力）+ tiktoken（BPE 分词）。
**不加载预训练权重**（避免下载 500MB+ 模型），仅用 config + tokenizer 做架构分析。

## 0. 环境准备
首次运行需安装依赖（取消注释执行一次）：

> ⚠️ transformers 仅用 config + tokenizer（秒级加载），不下载模型权重。
> tiktoken 和 torch 是纯本地库，无需 API key。

In [ ]:
# !pip install transformers torch tiktoken -q

## 1. 场景背景与营销映射

**核心命题**：LLM 是营销 Agent 的引擎。理解 Transformer 如何生成营销文案、token 成本如何影响推理成本。

**本上机解决**：
- 营销文案的中英文 token 消耗差异有多大？（直接影响推理成本）
- GPT-2 的架构参数有哪些？模型有多大？（理解 Scale Law）
- Self-Attention 如何关联营销文案中的关键词？（理解生成过程）

| TODO | 任务 | 真实库 | 营销映射 |
|------|------|--------|---------|
| TODO1 | Tokenization 对比 | tiktoken + transformers | 营销文案 token 成本 |
| TODO2 | GPT-2 架构分析 | transformers AutoConfig | 模型规模理解 |
| TODO3 | 手写 Self-Attention | torch | 注意力机制理解 |
| TODO4 | Multi-Head + TransformerBlock | torch | 完整架构块 |
| TODO5 | 注意力可视化 | torch + tiktoken | 营销关键词关联 |
| TODO6 | CLM 前向传播 + 训练阶段 | torch | 预训练任务理解 |

In [ ]:
import torch
import torch.nn.functional as F
import math
import tiktoken
from transformers import AutoConfig, AutoTokenizer

# ============================================================
# 营销文案语料（真实数据，基于电商场景）
# ============================================================

MARKETING_TEXTS = {
    "en_brief": "Write a Xiaohongshu marketing copy for a niacinamide serum targeting women aged 25-35",
    "zh_brief": "为一款烟酰胺精华液写小红书种草文案，目标人群25-35岁女性",
    "en_product": "Niacinamide Brightening Serum 5 percent niacinamide brightens skin tone and minimizes pores",
    "zh_product": "烟酰胺亮肤精华液 5%烟酰胺 提亮肤色 收缩毛孔",
}

# 模型定价表（$/token，基于 OpenAI 2026 定价）
MODEL_PRICING = {
    "gpt-4o": {"input": 2.50e-6, "output": 10.00e-6},
    "gpt-4o-mini": {"input": 0.15e-6, "output": 0.60e-6},
}

print("环境初始化完成")
print(f"transformers + torch + tiktoken 已加载")
print(f"营销文案样本: {len(MARKETING_TEXTS)} 条")
print(f"模型定价: {list(MODEL_PRICING.keys())}")

## TODO 1：用 tiktoken + transformers 对营销文案做 Tokenization

**tiktoken** 是 OpenAI 的 BPE 分词器，**transformers AutoTokenizer** 是 HuggingFace 的分词器接口。

**关键对比**：
- 英文 ~1 token ≈ 0.75 单词
- 中文 1 汉字 ≈ 1-2 token（BPE 主要在英文数据上训练）
- 这直接影响 LLM API 的 token 计费和推理成本

**BPE 子词可视化**：`enc.decode([token_id])` 可查看每个子词 token 的文本。高频词是完整 token，低频词被拆分为子词。

In [ ]:
# 用 tiktoken 和 transformers tokenizer 分别编码营销文本
enc_tiktoken = tiktoken.get_encoding('gpt2')
tok_hf = AutoTokenizer.from_pretrained('gpt2')

# 对比中英文 token 消耗
print("=== Tokenization 对比（tiktoken vs transformers）===")
for name, text in MARKETING_TEXTS.items():
    tt = len(enc_tiktoken.encode(text))
    ht = len(tok_hf.encode(text))
    print(f"  {name}: tiktoken={tt} tokens, transformers={ht} tokens, chars={len(text)}")

# BPE 子词可视化
print("\n=== BPE 子词可视化 ===")
for word in ['marketing', 'analytics', 'niacinamide', 'serum']:
    toks = enc_tiktoken.encode(word)
    decoded = [enc_tiktoken.decode([t]) for t in toks]
    print(f"  {word!r:16} -> {len(toks)} tokens: {decoded}")

# 推理成本计算
print("\n=== 推理成本计算 ===")
prompt = MARKETING_TEXTS['zh_brief']
response = '姐妹们！这款烟酰胺亮肤精华液真的绝了！主打5%烟酰胺，提亮肤色、收缩毛孔，效果看得见！价格才199元/30ml，性价比拉满！'
pt = len(enc_tiktoken.encode(prompt))
rt = len(enc_tiktoken.encode(response))
print(f"  Prompt: {pt} tokens | Response: {rt} tokens | Total: {pt+rt} tokens")
for model, rates in MODEL_PRICING.items():
    cost = pt * rates['input'] + rt * rates['output']
    daily = cost * 10000
    print(f"  {model}: ${cost:.6f}/次 -> 日均万次=${daily:.2f}")

## TODO 2：从 GPT-2 Config 读取架构参数，推算参数量

**AutoConfig.from_pretrained("gpt2")** 秒级加载 GPT-2 架构参数（不下载权重）：
- `n_layer`：Transformer Block 层数
- `n_head`：多头注意力的头数
- `n_embd`：嵌入维度
- `vocab_size`：词表大小
- `n_positions`：最大序列长度

**参数量推算**：
- Token embedding：`vocab_size × n_embd`
- Position embedding：`n_positions × n_embd`
- 每个 Block：attention（4 × n_embd²）+ FFN（8 × n_embd²）+ biases
- 总计 ≈ 124M（GPT-2 small）

**Scale Law 启示**：模型性能随参数量、数据量、计算量可预测地提升。

In [ ]:
# 从 GPT-2 config 读取架构参数（不加载权重）
config = AutoConfig.from_pretrained('gpt2')

print("=== GPT-2 Small 架构参数 ===")
print(f"  n_layer (Transformer Block 层数): {config.n_layer}")
print(f"  n_head (注意力头数): {config.n_head}")
print(f"  n_embd (嵌入维度): {config.n_embd}")
print(f"  vocab_size (词表大小): {config.vocab_size}")
print(f"  n_positions (最大序列长度): {config.n_positions}")

# 从 config 推算参数量
emb_params = config.vocab_size * config.n_embd + config.n_positions * config.n_embd
per_layer = (4 * config.n_embd * config.n_embd  # Q/K/V/O 投影
             + 8 * config.n_embd * config.n_embd  # FFN (up + down)
             + 4 * config.n_embd + 8 * config.n_embd)  # biases
total = emb_params + per_layer * config.n_layer + config.n_embd  # + final LN

print(f"\n=== 参数量推算 ===")
print(f"  Token embedding: {config.vocab_size * config.n_embd / 1e6:.1f}M")
print(f"  Position embedding: {config.n_positions * config.n_embd / 1e6:.1f}M")
print(f"  每个 Block: {per_layer / 1e6:.1f}M")
print(f"  总计 (approx): {total / 1e6:.1f}M (GPT-2 small ~124M)")
print(f"\n  Scale Law: GPT-2 small (124M) -> GPT-2 XL (1.5B) -> GPT-3 (175B)")
print(f"  模型每增大 10x，能力显著提升，但推理成本也线性增长")

## TODO 3：手写 Self-Attention

**Self-Attention 公式**：

```
Attention(Q, K, V) = softmax(Q × K^T / √d_k) × V
```

**计算步骤**：
1. Q = x @ W_q, K = x @ W_k, V = x @ W_v（线性投影）
2. scores = Q @ K^T / √d_k（点积 + 缩放）
3. attn = softmax(scores)（归一化为概率分布，行和为 1）
4. out = attn @ V（加权聚合 Value）

**为什么除以 √d_k**：防止点积值过大导致 softmax 梯度消失。

In [ ]:
def self_attention(x, W_q, W_k, W_v):
    """单头 Self-Attention（教学版）
    x: (seq, d), W_q/W_k/W_v: (d, d)
    返回: out (seq, d), attn (seq, seq)
    """
    Q = x @ W_q  # (seq, d)
    K = x @ W_k  # (seq, d)
    V = x @ W_v  # (seq, d)
    d_k = K.size(-1)
    # 点积 + 缩放（防止 softmax 梯度消失）
    scores = Q @ K.transpose(-2, -1) / math.sqrt(d_k)  # (seq, seq)
    # 归一化为概率分布（每行和为 1）
    attn = F.softmax(scores, dim=-1)  # (seq, seq)
    # 加权聚合 Value
    out = attn @ V  # (seq, d)
    return out, attn

# 测试
torch.manual_seed(42)
seq, d = 6, 8
x = torch.randn(seq, d)
W_q = torch.randn(d, d)
W_k = torch.randn(d, d)
W_v = torch.randn(d, d)

out, attn = self_attention(x, W_q, W_k, W_v)
print(f"=== Self-Attention 测试 ===")
print(f"  Input shape: {x.shape} (seq={seq}, d={d})")
print(f"  Output shape: {out.shape}")
print(f"  Attention shape: {attn.shape} (seq x seq)")
print(f"  Row sums (应为 1.0): {attn.sum(dim=-1).tolist()}")
print(f"  Attention matrix (前 4x4):")
for i in range(min(4, seq)):
    row = "  ".join(f"{attn[i][j]:.3f}" for j in range(min(4, seq)))
    print(f"    [{row}  ...]")

## TODO 4：Multi-Head Attention + Transformer Block

**Multi-Head Attention**：将 Q/K/V 分成 `n_heads` 组，每组独立计算 Attention，然后拼接。

**Transformer Block 完整结构**：
```
输入
  ├─ Multi-Head Self-Attention
  |    └─ 残差连接 + LayerNorm
  ├─ Feed-Forward Network (FFN, 4x 扩展)
  |    └─ 残差连接 + LayerNorm
输出
```

- **残差连接**：`x = LayerNorm(x + f(x))`，解决深层网络梯度消失
- **FFN**：`Linear(d, 4d) → GELU → Linear(4d, d)`，非线性变换
- **LayerNorm**：归一化每层输出，稳定训练

In [ ]:
class TransformerBlock(torch.nn.Module):
    """Transformer Block: Multi-Head Attention + FFN + 残差 + LayerNorm"""
    def __init__(self, d, n_heads):
        super().__init__()
        self.n_heads = n_heads
        self.W_q = torch.nn.Linear(d, d, bias=False)
        self.W_k = torch.nn.Linear(d, d, bias=False)
        self.W_v = torch.nn.Linear(d, d, bias=False)
        self.W_o = torch.nn.Linear(d, d, bias=False)
        self.ffn = torch.nn.Sequential(
            torch.nn.Linear(d, d * 4),   # 扩展 4 倍
            torch.nn.GELU(),              # 激活函数
            torch.nn.Linear(d * 4, d),    # 压缩回来
        )
        self.ln1 = torch.nn.LayerNorm(d)
        self.ln2 = torch.nn.LayerNorm(d)

    def forward(self, x):
        # x: (batch, seq, d)
        B, S, D = x.shape
        d_h = D // self.n_heads
        # Multi-Head: reshape 为 (batch, heads, seq, d_h)
        Q = self.W_q(x).view(B, S, self.n_heads, d_h).transpose(1, 2)
        K = self.W_k(x).view(B, S, self.n_heads, d_h).transpose(1, 2)
        V = self.W_v(x).view(B, S, self.n_heads, d_h).transpose(1, 2)
        # Scaled dot-product attention
        scores = Q @ K.transpose(-2, -1) / math.sqrt(d_h)
        attn = F.softmax(scores, dim=-1)
        out = attn @ V  # (batch, heads, seq, d_h)
        # 拼接所有 head
        out = out.transpose(1, 2).contiguous().view(B, S, D)
        out = self.W_o(out)
        # 残差 + LayerNorm
        x = self.ln1(x + out)
        # FFN + 残差 + LayerNorm
        x = self.ln2(x + self.ffn(x))
        return x, attn

# 测试
torch.manual_seed(42)
block = TransformerBlock(d=8, n_heads=4)
x_batch = torch.randn(1, 6, 8)  # (batch=1, seq=6, d=8)
out, attn = block(x_batch)
print(f"=== Transformer Block 测试 ===")
print(f"  Input: {x_batch.shape} (batch=1, seq=6, d=8)")
print(f"  Output: {out.shape}")
print(f"  Attention: {attn.shape} (batch, heads, seq, seq)")
print(f"  Block params: {sum(p.numel() for p in block.parameters())}")
print(f"  n_heads=4, d_h=2 (每个头看 2 维)")
print(f"  FFN 扩展: 8 -> 32 -> 8 (4x 扩展)")

## TODO 5：在营销文案 Token 上可视化注意力

将营销文案 token化后，用 TODO3 的 Self-Attention 计算注意力矩阵，查看哪些词相互关联最强。

**步骤**：
1. 用 tiktoken 将英文产品描述 token 化
2. 为每个 token 生成随机 embedding（教学版，真实模型用学习的 embedding）
3. 运行 Self-Attention
4. 打印注意力矩阵，找出每个 token 最关注的其他 token

**营销洞察**：注意力矩阵显示模型在生成文案时"看"哪些词。例如"niacinamide"可能强烈关注"serum"和"brightens"。

In [ ]:
# 在营销文案 token 上可视化注意力
enc = tiktoken.get_encoding('gpt2')
text = MARKETING_TEXTS['en_product']
tokens = enc.encode(text)
subwords = [enc.decode([t]) for t in tokens]

print(f"=== 营销文案注意力可视化 ===")
print(f"Text: {text}")
print(f"Tokens ({len(tokens)}): {subwords}")

# 为每个 token 生成随机 embedding（教学版）
# 真实模型用学习的 Embedding 层
torch.manual_seed(42)
d = 16
seq = len(tokens)
x = torch.randn(seq, d)
W_q = torch.randn(d, d)
W_k = torch.randn(d, d)
W_v = torch.randn(d, d)

# 运行 Self-Attention
_, attn = self_attention(x, W_q, W_k, W_v)

# 打印注意力矩阵
print(f"\n注意力矩阵 ({seq}x{seq}):")
header = "        " + "  ".join(f"{s[:5]:>5s}" for s in subwords)
print(header)
for i in range(seq):
    row = "  ".join(f"{attn[i][j]:.2f}" for j in range(seq))
    print(f"  {subwords[i][:5]:>5s}  {row}")

# 每个 token 最关注的 top-2
print(f"\n每个 token 最关注的 top-2:")
for i in range(seq):
    top2 = attn[i].topk(2)
    attended = [(subwords[j], f"{v:.3f}") for j, v in zip(top2.indices, top2.values)]
    print(f"  {subwords[i]!r:16} -> {attended}")

print(f"\n注意: 随机 embedding 的注意力无语义意义（教学版）。")
print(f"真实模型用预训练 embedding，'niacinamide' 会关注 'serum'/'brightens'。")

## TODO 6：CLM 前向传播 + 训练三阶段

**CLM（Causal Language Modeling）**：预训练的核心任务--给定前面的 token，预测下一个 token。

```
输入:  [t0, t1, t2, ..., t_{n-1}]
目标:  [t1, t2, t3, ..., t_n]
```

**实现 MiniGPT**：Token Embedding + Position Embedding + N 个 TransformerBlock + 输出投影。

**训练三阶段**：
1. **Pre-training**：海量文本预测下一 token → Base Model
2. **SFT**：指令-回答对监督微调 → Chat Model
3. **Alignment**：RLHF/DPO 对齐人类偏好 → Aligned Model

本 TODO 演示 CLM 前向传播（不训练，仅展示 logits 和 loss 计算方式）。

In [ ]:
class MiniGPT(torch.nn.Module):
    """简化版 GPT: Embedding + N x TransformerBlock + 输出投影"""
    def __init__(self, vocab_size, d=32, n_heads=4, n_layers=2, max_seq=100):
        super().__init__()
        self.tok_emb = torch.nn.Embedding(vocab_size, d)
        self.pos_emb = torch.nn.Embedding(max_seq, d)
        self.blocks = torch.nn.ModuleList([
            TransformerBlock(d, n_heads) for _ in range(n_layers)
        ])
        self.ln = torch.nn.LayerNorm(d)
        self.head = torch.nn.Linear(d, vocab_size, bias=False)

    def forward(self, idx):
        # idx: (batch, seq)
        B, S = idx.shape
        pos = torch.arange(S).unsqueeze(0)
        x = self.tok_emb(idx) + self.pos_emb(pos)
        attns = []
        for block in self.blocks:
            x, a = block(x)
            attns.append(a)
        x = self.ln(x)
        logits = self.head(x)  # (batch, seq, vocab)
        return logits, attns

# CLM 前向传播演示
enc = tiktoken.get_encoding('gpt2')
text = MARKETING_TEXTS['en_brief']
tokens = enc.encode(text)
idx = torch.tensor([tokens])

torch.manual_seed(42)
model = MiniGPT(vocab_size=enc.n_vocab, d=32, n_heads=4, n_layers=2)
logits, attns = model(idx)

print(f"=== CLM 前向传播（预训练任务演示）===")
print(f"  输入: {len(tokens)} tokens")
print(f"  Logits shape: {logits.shape} (batch, seq, vocab)")
print(f"  Model params: {sum(p.numel() for p in model.parameters()) / 1e3:.1f}K")
print(f"  Attention layers: {len(attns)}")

# CLM Loss: 预测下一个 token
# logits[:, :-1] 预测 tokens[:, 1:]
shift_logits = logits[:, :-1, :]
shift_targets = idx[:, 1:]
loss = F.cross_entropy(
    shift_logits.reshape(-1, shift_logits.size(-1)),
    shift_targets.reshape(-1)
)
print(f"  CLM loss (未训练): {loss.item():.4f}")
print(f"  Perplexity: {torch.exp(loss).item():.1f} (未训练，应接近 vocab_size)")

# 训练三阶段概述
print(f"\n=== LLM 训练三阶段 ===")
stages = [
    ("1. Pre-training", "预测下一 token", "数万亿 token", "Base Model (有知识不会对话)"),
    ("2. SFT", "指令-回答对微调", "数万-数十万条", "Chat Model (能遵循指令)"),
    ("3. Alignment", "RLHF/DPO 对齐", "偏好排序数据", "Aligned Model (安全+有用)"),
]
for stage, task, data, output in stages:
    print(f"  {stage}: {task}")
    print(f"    数据: {data} | 产出: {output}")

print(f"\n=== 营销映射 ===")
print(f"  Pre-training: 模型学会语言 + 营销知识")
print(f"  SFT: 模型学会'写小红书种草文案'指令")
print(f"  Alignment: 模型不生成虚假宣传/违禁词")

## 3. 反思与前沿

### 反思问题
1. 营销文案的中英文 token 消耗差异有多大？对推理成本有什么影响？
2. GPT-2 small 有 124M 参数，GPT-3 有 175B--参数量增长 1000 倍带来什么能力提升？推理成本呢？
3. Self-Attention 的注意力矩阵中，对角线值通常较大，为什么？（每个 token 最关注自己）
4. 预训练用"预测下一 token"这么简单的任务，为什么能训练出如此强大的模型？

### 2026 前沿：推理成本优化
- **DeepSeek-MoE**（arXiv 2401.04088）：MoE 架构，671B 总参数 / 37B 激活参数，推理成本远低于 Dense 模型
- **投机解码**（arXiv 2211.17192）：小模型生成候选 token，大模型并行验证，延迟降低 2-3x
- **vLLM**（https://github.com/vllm-project/vllm）：PagedAttention + 连续批处理，吞吐量 14-24x
- **多模态 + 对比学习**：CLIP 用对比损失对齐图文，GPT-4o 端到端多模态训练

参考 [DeepSeek-MoE](https://arxiv.org/abs/2401.04088) + [投机解码论文](https://arxiv.org/abs/2211.17192) + [vLLM](https://github.com/vllm-project/vllm)。